In [ ]:
import sys, os
from pathlib import Path
_root = Path(os.getcwd())
while not ((_root / 'src').exists() and (_root / 'pyproject.toml').exists()) and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

# Atenção texto × séries temporais: sinal ou apenas cobertura?

**Pergunta:** manchetes verificadas acrescentam sinal à previsão numérica de retornos relativos? Esta é uma demonstração de pesquisa, **não uma estratégia com alfa comprovado**. O notebook apresenta a atenção residual já implementada e executada, usa embeddings MOMENT-1-small e MiniLM gerados na execução anterior e lê resultados existentes para abrir rapidamente no Colab. Se os resultados ainda não existirem, treina os dois resíduos reais em uma GPU; nunca inventa métricas.

**Roteiro:** proveniência → disponibilidade das notícias → arquitetura → ablações → validação → previsão e carteira → diagnóstico textual → perfis simulados → sensibilidade a custos → controles de manchetes → regimes HMM causais → disponibilidade de dados para um período novo → limitações.

In [ ]:
import json, subprocess
import numpy as np
import pandas as pd
from IPython.display import Image, display
from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/latent_fusion')
if REPO.exists():
    if not (REPO / '.git').is_dir():
        raise RuntimeError(f'Pasta existente não é repositório Git: {REPO}')
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/rbfcdog/latent_fusion.git', str(REPO)], check=True)
sys.path.insert(0, str(REPO))
DATA = Path('/content/drive/MyDrive/latent_fusion_data')
SOURCE = DATA / 'fast_fusion_demo_results'
for filename in ('fast_demo_panel.npz', 'fast_demo_events.npz', 'fast_demo_provenance.json', 'ts_only.pt', 'demo_titles.jsonl.gz'):
    if not (SOURCE / filename).is_file():
        raise FileNotFoundError(f'Execute antes fast_fusion_demo_colab.ipynb: {SOURCE / filename}')
{'fonte': str(SOURCE), 'repositório': str(REPO)}

## 1. Dados, período e rastreabilidade

O painel guarda **retornos observados e embeddings numéricos gerados naquela execução**; o arquivo de eventos guarda embeddings de manchetes e atributos léxicos. As manchetes têm a data de decisão atrasada em um pregão quando o horário de publicação não é seguro. A validação abaixo confere que o painel, os eventos e o checkpoint numérico pertencem à mesma execução. Nenhum encoder, LLM ou notícia adicional é baixado.

In [ ]:
from src.fusion.fast_residual_experiment import load_saved_demo, run_experiment
from src.fusion.text_attention import TextEventDataset
panel, events, baseline_checkpoint, provenance = load_saved_demo(SOURCE)
display(pd.DataFrame([{'fonte numérica': provenance['moment_model_id'], 'fonte textual': provenance['text_model_id'], 'embeddings numéricos': provenance['moment_embeddings'], 'manchetes': provenance['article_embeddings'], 'ativos': len(panel['tickers']), 'geração UTC': provenance['created_at_utc']}]))
assert provenance['fresh_inference'] and not provenance['old_embedding_cache_used']

## 2. Alvo e informação disponível

Para cada ativo e fechamento de decisão $t$, o alvo é o retorno dos próximos **cinco pregões** menos a média dos retornos futuros dos mesmos oito ativos. Esse retorno futuro é **somente rótulo/avaliação**, nunca entrada do modelo. As entradas são os últimos 64 retornos já conhecidos e manchetes disponíveis até $t$, no máximo oito das últimas cinco sessões. Treino, validação e teste são cronológicos, com purga dos rótulos de cinco dias nas fronteiras. O teste usa todos os pregões, inclusive os sem manchete; ‘com notícia’ não significa notícia publicada no próprio dia.

In [ ]:
coverage = []
for split in ('train', 'val', 'test'):
    dataset = TextEventDataset(panel, events, split)
    dates = [panel['arrays'][int(ti)]['dates'][int(j)] for ti, j in dataset.samples]
    with_news = sum(bool(dataset._news(int(ti), int(j))[1].any()) for ti, j in dataset.samples)
    coverage.append({'partição': split, 'primeira decisão': str(pd.Timestamp(min(dates)).date()), 'última decisão': str(pd.Timestamp(max(dates)).date()), 'amostras': len(dataset), 'com notícia nos últimos 5 pregões': with_news, 'cobertura (%)': 100 * with_news / len(dataset)})
coverage = pd.DataFrame(coverage).set_index('partição')
display(coverage.round(2))

## 3. Estratégia: atenção cruzada residual

O **modelo numérico pré-treinado fica congelado**. Sua representação das séries temporais é a consulta de uma `MultiheadAttention` de quatro cabeças; chaves/valores são até oito manchetes elegíveis e uma chave nula válida. O ramo textual usa o MiniLM mais atributos léxicos, novidade e idade. Uma porta sigmoidal modula a correção limitada: $\hat y_t = \hat y_t^{\mathrm{num}} + I_{\mathrm{notícia}} \; 0{,}02 \;\sigma(g_t)\tanh(r_t)$. A última camada do residual começa em zero: antes do treino a previsão é exatamente a numérica. Sem notícia recente, o residual é zero.

**Três controles no mesmo calendário:** numérico congelado; numérico + atenção apenas a idade/contagem das manchetes; numérico + atenção a conteúdo, atributos e idade. A comparação texto × idade/contagem é a ablação necessária para investigar **informação semântica**, em vez de confundir presença de notícia com significado. A previsão de excesso zero é referência de erro, não carteira.

## 4. Treino ou reutilização auditada

Se houver resultados completos com **a mesma proveniência** e custo de 10 bps, o notebook os lê imediatamente (inclusive em Colab CPU). Caso contrário, treina os dois resíduos na GPU, em saída separada, sem recalcular embeddings ou descongelar o modelo numérico. Somente o período de validação escolhe a época; o teste 2022–23 não participa da escolha.

In [ ]:
required = ('run_config.json', 'signal_metrics.csv', 'portfolio_metrics.csv', 'equity_curves.csv', 'metadata_history.csv', 'text_residual_history.csv', 'predictions_ts_only.csv', 'predictions_metadata.csv', 'predictions_text_residual.csv', 'residual_vs_bh.png')
candidates = (DATA / 'fast_residual_attention_results', DATA / 'attention_presentation_results')
ready = [folder for folder in candidates if all((folder / name).is_file() for name in required)]
if ready:
    OUT = ready[0]
    mode = 'Resultados reais da execução anterior'
else:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Sem resultados completos: ative T4/L4 no Colab para treinar o residual')
    OUT = candidates[1]
    run_experiment(SOURCE, OUT, cost_bps=10)
    mode = 'Treino real concluído nesta execução'
config = json.loads((OUT / 'run_config.json').read_text(encoding='utf-8'))
if config['embedding_provenance'] != provenance or config['cost_bps'] != 10 or config['models'] != ['ts_only', 'metadata', 'text_residual', 'zero', 'buy_hold']:
    raise ValueError('Resultados incompatíveis com os embeddings e controles carregados')
{'modo': mode, 'diretório': str(OUT), 'épocas máximas': config['max_epochs'], 'residual máximo': config['max_residual']}

## 5. Diagnóstico de validação

O erro de validação considera **somente linhas com manchete**, pois dias sem notícia devolvem o modelo numérico congelado. O gráfico permite verificar se continuar treinando melhora o treino mas piora a validação; não compare seu valor diretamente ao histórico numérico calculado em outra população de linhas.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src.backtest.visualization import BG, PANEL, CYAN, PURPLE, WHITE
histories = {name: pd.read_csv(OUT / f'{name}_history.csv') for name in ('metadata', 'text_residual')}
fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor(PANEL)
for name, label, color in (('metadata', 'Idade/contagem', PURPLE), ('text_residual', 'Manchetes', CYAN)):
    history = histories[name]
    ax.plot(history['epoch'], history['val_mse'], marker='o', linewidth=2.2, label=label, color=color)
ax.set_title('Erro de validação: resíduos com notícias', fontsize=16, color='white')
ax.set_xlabel('Época', fontsize=14, color='white')
ax.set_xticks(sorted(set(histories['metadata']['epoch']) | set(histories['text_residual']['epoch'])))
ax.set_ylabel('MSE na validação', fontsize=14, color='white')
ax.tick_params(colors='white', labelsize=12)
ax.grid(alpha=0.12)
ax.legend(facecolor=PANEL, labelcolor='white', edgecolor=WHITE, fontsize=14)
fig.tight_layout()
fig.savefig(OUT / 'presentation_validation.png', dpi=300, facecolor=BG, edgecolor='none')
display(fig)
plt.close(fig)
{'melhor época (metadados)': int(histories['metadata'].loc[histories['metadata']['val_mse'].idxmin(), 'epoch']), 'melhor época (texto)': int(histories['text_residual'].loc[histories['text_residual']['val_mse'].idxmin(), 'epoch'])}

## 6. Evidência fora da amostra: previsão e carteira

MSE, MAE e IC Spearman avaliam o retorno relativo futuro de cinco pregões. A carteira transforma as previsões em pesos **long-only** e rebalanceia a cada cinco pregões após o fechamento. Retornos são líquidos de 10 bps por valor negociado, inclusive entrada; a referência compra os mesmos oito ativos com pesos iguais e mantém. Compare o texto também à idade/contagem: superar apenas uma fusão anterior que reaprendeu a cabeça numérica **não prova** contribuição da linguagem.

In [ ]:
signal = pd.read_csv(OUT / 'signal_metrics.csv', index_col=0)
portfolio = pd.read_csv(OUT / 'portfolio_metrics.csv', index_col=0)
equity = pd.read_csv(OUT / 'equity_curves.csv', parse_dates=['date']).set_index('date')
assert len(equity) > 0 and all(np.isclose(equity[name].iloc[-1], 1 + portfolio.loc[name, 'return_pct'] / 100) for name in portfolio.index)
display(signal[['mse_5d', 'mae_5d', 'ic_spearman', 'n_amostras', 'com_evento']].round(8))
display(portfolio[['return_pct', 'bh_return_pct', 'excess_pct', 'alpha_annual_pct', 'beta', 'sharpe', 'turnover_total']].round(6))
display(Image(filename=str(OUT / 'residual_vs_bh.png')))

## 7. Quanto a atenção realmente mudou?

Uma curva visualmente próxima à referência pode esconder um efeito ínfimo. Abaixo, as previsões são alinhadas por **ticker e data**; a correção média absoluta e a quantidade de pregões cujo ativo de maior pontuação mudou medem a materialidade do residual, sem reinterpretar pesos de atenção como causalidade. Tolerância de $10^{-6}$ em dias sem notícia permite diferenças numéricas de GPU entre tamanhos de lote.

In [ ]:
forecasts = {name: pd.read_csv(OUT / f'predictions_{name}.csv', parse_dates=['date']) for name in ('ts_only', 'metadata', 'text_residual')}
reference = forecasts['ts_only']
if (SOURCE / 'predictions_ts_only.csv').is_file():
    original = pd.read_csv(SOURCE / 'predictions_ts_only.csv', parse_dates=['date'])
    if not original[['ticker', 'date']].equals(reference[['ticker', 'date']]) or not np.allclose(original['score'], reference['score'], atol=1e-6):
        raise ValueError('O checkpoint numérico não reproduz as previsões da execução original')
diagnostics = []
leaders = reference.loc[reference.groupby('date')['score'].idxmax()].set_index('date')['ticker']
for name in ('metadata', 'text_residual'):
    frame = forecasts[name]
    if not frame[['ticker', 'date']].equals(reference[['ticker', 'date']]) or not frame['has_event'].equals(reference['has_event']):
        raise ValueError('A ablação mudou as datas ou a cobertura do teste')
    delta = frame['score'] - reference['score']
    if delta.loc[~reference['has_event']].abs().max() > 1e-6:
        raise ValueError('O residual modificou uma previsão sem notícia')
    current_leaders = frame.loc[frame.groupby('date')['score'].idxmax()].set_index('date')['ticker']
    diagnostics.append({'modelo': name, 'correção média absoluta': delta.abs().mean(), 'correção máxima absoluta': delta.abs().max(), 'pregões com líder diferente': int((leaders != current_leaders).sum())})
display(pd.DataFrame(diagnostics).set_index('modelo').round(8))
{'diferença texto − metadados (pontos percentuais)': round(float(portfolio.loc['text_residual', 'return_pct'] - portfolio.loc['metadata', 'return_pct']), 6), 'MSE da previsão zero': float(signal.loc['zero', 'mse_5d'])}

## 8. Perfis de investidor: pesos e frequência (simulação ilustrativa)

O `TODO.md` pede perfis e rebalanceamento de longo prazo. **Sem treinar novamente nem mudar a previsão**, aplicamos três regras pré-definidas à mesma pontuação de cada modelo: conservador (teto de 18% por ativo, inclinação 0,5×, rebalanceamento a cada **63 pregões** ≈ trimestre); moderado (25%, 1×, **21 pregões** ≈ mês); arrojado (40%, 2×, **5 pregões**). Os pesos são long-only, sempre somam 100% nos **mesmos oito ativos**, sem alavancagem ou caixa. O teto redistribui qualquer excesso entre os ativos restantes; **não é um limite de risco ou de perda**. Estes perfis de *carteira de oito ativos* não são os `InvestorProfile` de posições únicas do outro backtest.

Todos começam no **mesmo dia**, terminam no **mesmo fechamento**, pagam 10 bps sobre o valor negociado (inclusive entrada) e comparam com **o mesmo comprar e manter**. Nada foi escolhido para maximizar os retornos de 2022–23; as regras são apenas cenários ilustrativos definidos antes desta comparação. Altere `PROFILE` para escolher a visualização; a tabela compara todos os perfis e as três variantes numéricas/textuais. Resultados não substituem avaliação de adequação real do investidor, impostos ou execução.


In [ ]:
from src.fusion.presentation_profiles import PROFILES, compare_profiles, profile_weights
PROFILE = 'moderado'
if PROFILE not in PROFILES:
    raise ValueError('Escolha conservador, moderado ou arrojado')
profile_specs = pd.DataFrame([{'perfil': name, 'teto por ativo (%)': 100 * rule['max_position'], 'inclinação relativa': rule['strength'], 'intervalo (pregões)': rule['sessions']} for name, rule in PROFILES.items()]).set_index('perfil')
comparison, profile_curves, first_allocations, profile_bh = compare_profiles(panel, forecasts, equity.index[-1], cost_bps=10)
if not profile_curves.index.equals(equity.index) or not np.allclose(profile_bh, equity['buy_hold']):
    raise ValueError('Perfil não usa o mesmo período e comprar e manter da avaliação original')
first_weights = first_allocations.xs('text_residual', level='modelo')['peso_pct'].unstack('perfil').reindex(panel['tickers'])
first_weights['comprar e manter'] = 100 / len(panel['tickers'])
display(profile_specs, first_weights.round(2))
display(comparison[['return_pct', 'excess_pct', 'alpha_annual_pct', 'beta', 'sharpe', 'queda_máxima_pct', 'turnover_total', 'rebalanceamentos']].round(3))


## 9. Perfil selecionado: evolução, primeira e última alocação

`PROFILE` altera **intensidade, teto e intervalo de rebalanceamento**, mas não modifica o checkpoint de atenção. A tabela abaixo mostra pesos hipotéticos no primeiro e no último rebalanceamento do mesmo perfil para o **sinal com texto**; nenhum retorno futuro entra no peso. O gráfico mostra a carteira numérica, o controle idade/contagem, o texto e o comprar e manter **nesse perfil específico**. Comparar perfis diferentes pelo retorno final nesta mesma amostra já vista **não autoriza escolher o melhor para investir**.


In [ ]:
policy = PROFILES[PROFILE]
first_date = pd.Timestamp(equity.index[0])
eligible = pd.DatetimeIndex(forecasts['text_residual']['date'].unique()).intersection(equity.index[:-5]).sort_values()
last_rebalance = eligible[::policy['sessions']][-1]
allocation_snapshots = {}
for label, date in (('primeiro', first_date), ('último', last_rebalance)):
    decision = forecasts['text_residual'].loc[forecasts['text_residual']['date'] == date].set_index('ticker')['score'].reindex(panel['tickers'])
    if decision.isna().any():
        raise ValueError('Ativo sem previsão para demonstrar pesos')
    allocation_snapshots[f'{label} ({date.date()})'] = profile_weights(decision.to_numpy(), policy) * 100
display(pd.DataFrame(allocation_snapshots, index=panel['tickers']).round(2))
from src.backtest.visualization import BG, PANEL, GOLD, PURPLE, CYAN, GREEN, WHITE
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor(PANEL)
for model, label, color in (('ts_only', 'Somente série temporal', GOLD), ('metadata', 'Idade/contagem', PURPLE), ('text_residual', 'Manchetes + atenção', CYAN)):
    ax.plot(profile_curves.index, profile_curves[(PROFILE, model)], label=label, color=color, linewidth=2.2)
ax.plot(profile_bh.index, profile_bh, label='Comprar e manter', color=GREEN, linestyle='--', linewidth=2.2)
ax.axhline(1, color=WHITE, alpha=0.2, linestyle='--')
ax.set_title(f'Perfil {PROFILE}: capital após custos, período idêntico', fontsize=16, color='white')
ax.set_ylabel('Capital (base 1; após custos)', fontsize=14, color='white')
ax.tick_params(colors='white', labelsize=12)
ax.grid(alpha=0.12)
ax.legend(facecolor=PANEL, labelcolor='white', edgecolor=WHITE, fontsize=14)
fig.tight_layout()
fig.savefig(OUT / f'presentation_profile_{PROFILE}.png', dpi=300, facecolor=BG, edgecolor='none')
display(fig)
plt.close(fig)


## 10. Custos e pior período: viabilidade antes de retorno

Do `TODO.md`, esta seção reaproveita **custo operacional**, **pior caso** e **comparação com o mercado** sem criar trades fictícios. Reexecuta o mesmo backtest com custos hipotéticos de 0, 10 e 25 bps para o `PROFILE` escolhido, **sem reestimar modelos ou trocar sinais**. São custos por valor negociado, não incluem impostos, slippage adicional ou capacidade de execução. A pior variação mensal é apenas um diagnóstico deste teste, não um cenário de estresse futuro. A queda máxima é mostrada na tabela dos perfis.


In [ ]:
from src.fusion.event_experiment import _portfolio, _portfolio_metrics
cost_rows = []
for bps in (0, 10, 25):
    for model in ('ts_only', 'metadata', 'text_residual'):
        curve = _portfolio(panel, forecasts[model], bps, rebalance_sessions=policy['sessions'], weight_fn=lambda scores: profile_weights(scores, policy), end_date=equity.index[-1])
        metrics = _portfolio_metrics(curve, bps)
        cost_rows.append({'custo (bps)': bps, 'modelo': model, 'retorno líquido (%)': metrics['return_pct'], 'excesso vs BH (p.p.)': metrics['excess_pct'], 'giro total': metrics['turnover_total']})
display(pd.DataFrame(cost_rows).set_index(['custo (bps)', 'modelo']).round(3))
month_end = profile_curves[PROFILE].resample('M').last()
month_return = month_end.pct_change()
month_return.iloc[0] = month_end.iloc[0] - 1
month_return['comprar e manter'] = profile_bh.resample('M').last().pct_change()
month_return.iloc[0, month_return.columns.get_loc('comprar e manter')] = profile_bh.resample('M').last().iloc[0] - 1
worst_month = month_return['text_residual'].idxmin()
{'pior mês observado': str(worst_month.date()), 'retornos nesse mês (%)': (100 * month_return.loc[worst_month]).round(2).to_dict()}


## 11. Por ativo e por presença de notícia

O `TODO.md` também pede **análise por ticker** e comparação **com notícia / sem notícia**. Para cada ativo, a diferença de MSE entre metadados e texto indica onde o conteúdo melhorou ou piorou a previsão do retorno relativo de cinco pregões; valores sobrepostos no tempo e mesma amostra já examinada **não são significância estatística**. Uma manchete nos últimos cinco pregões não garante cobertura integral da empresa. Sem manchete recente, ambos os resíduos devem preservar o mesmo modelo numérico.


In [ ]:
ticker_diagnostics = []
for ticker, positions in reference.groupby('ticker').groups.items():
    observed = reference.loc[positions, 'excess_return_5d'].to_numpy()
    with_news = reference.loc[positions, 'has_event'].sum()
    mse_meta = np.mean((forecasts['metadata'].loc[positions, 'score'].to_numpy() - observed) ** 2)
    mse_text = np.mean((forecasts['text_residual'].loc[positions, 'score'].to_numpy() - observed) ** 2)
    ticker_diagnostics.append({'ativo': ticker, 'sessões com notícia recente': int(with_news), 'sessões no teste': len(positions), 'MSE só metadados': mse_meta, 'MSE com texto': mse_text, 'Δ MSE (metadados − texto)': mse_meta - mse_text})
display(pd.DataFrame(ticker_diagnostics).set_index('ativo').round(8))
news_groups = []
for present, subset in reference.groupby('has_event'):
    positions = subset.index
    observed = subset['excess_return_5d'].to_numpy()
    news_groups.append({'notícia nos últimos 5 pregões': bool(present), 'observações': len(subset), **{f'MSE {name}': np.mean((forecasts[name].loc[positions, 'score'].to_numpy() - observed) ** 2) for name in ('ts_only', 'metadata', 'text_residual')}})
display(pd.DataFrame(news_groups).set_index('notícia nos últimos 5 pregões').round(8))


## 12. Controles: manchetes embaralhadas e somente texto

O controle embaralhado troca **conteúdo**, não datas, quantidade de manchetes, idade nem máscara: cada manchete doadora estava disponível até a data de decisão da destinatária. Treina outro residual com as mesmas partições; a primeira manchete isolada pode ficar inalterada. O modelo **somente texto** agrega MiniLM, pistas léxicas/sentimento e novidade, sem retornos, MOMENT, idade ou identificador do ativo; a máscara ainda indica presença de manchete. Todos usam o mesmo teste já inspecionado em 2022–23, o mesmo comprar e manter e custos de 10 bps. Ganhos nesse teste continuam exploratórios, não são confirmação independente.

Se os controles já tiverem sido calculados com estes embeddings, carregamos os resultados. Caso contrário, **somente com GPU no Colab** reaproveitamos os checkpoints existentes, verificamos suas previsões e treinamos os dois controles novos; sem GPU mostramos explicitamente que os controles ainda não foram executados, sem fabricar números.

In [ ]:
CONTROL_OUT = DATA / 'attention_presentation_controls'
control_files = ('run_config.json', 'signal_metrics.csv', 'portfolio_metrics.csv', 'equity_curves.csv', 'paired_score_deltas.csv', 'shuffled_content_mapping.csv', 'predictions_ts_only.csv', 'predictions_metadata.csv', 'predictions_text_residual.csv', 'predictions_shuffled_content.csv', 'predictions_text_only.csv')
if all((CONTROL_OUT / name).is_file() for name in control_files):
    control_mode = 'Resultados reais de execução anterior'
elif all((CONTROL_OUT / name).is_file() for name in ('run_config.json', 'signal_metrics.csv', 'portfolio_metrics.csv')):
    raise ValueError('Controles parcialmente salvos: remova a saída incompleta e execute novamente com GPU')
else:
    import torch
    if torch.cuda.is_available():
        from src.fusion.presentation_controls import run_controls
        run_controls(SOURCE, CONTROL_OUT, cost_bps=10, previous_results=OUT)
        control_mode = 'Dois controles treinados nesta execução, checkpoints anteriores reaproveitados'
    else:
        control_mode = 'Controles ainda não executados; use uma GPU no Colab após liberar a outra sessão'
control_forecasts = {}
if all((CONTROL_OUT / name).is_file() for name in control_files):
    control_config = json.loads((CONTROL_OUT / 'run_config.json').read_text(encoding='utf-8'))
    if control_config['embedding_provenance'] != provenance or control_config['cost_bps'] != 10 or control_config['control_seed'] != 42 or control_config['baseline_seed'] != baseline_checkpoint['seed']:
        raise ValueError('Controles incompatíveis com os embeddings e custos carregados')
    for name in ('ts_only', 'metadata', 'text_residual', 'shuffled_content', 'text_only'):
        frame = pd.read_csv(CONTROL_OUT / f'predictions_{name}.csv', parse_dates=['date'])
        if not frame[['ticker', 'date', 'has_event', 'excess_return_5d']].equals(reference[['ticker', 'date', 'has_event', 'excess_return_5d']]):
            raise ValueError(f'O controle {name} mudou as decisões, rótulos ou cobertura')
        if name in forecasts and not np.allclose(frame['score'], forecasts[name]['score'], rtol=1e-5, atol=1e-6):
            raise ValueError(f'O controle {name} não reproduz a previsão original')
        if name not in forecasts:
            control_forecasts[name] = frame
    control_signal = pd.read_csv(CONTROL_OUT / 'signal_metrics.csv', index_col=0)
    control_portfolio = pd.read_csv(CONTROL_OUT / 'portfolio_metrics.csv', index_col=0)
    mapping = pd.read_csv(CONTROL_OUT / 'shuffled_content_mapping.csv')
    display(control_signal.loc[['ts_only', 'metadata', 'text_residual', 'shuffled_content', 'text_only', 'zero'], ['mse_5d', 'ic_spearman', 'n_amostras', 'com_evento']].round(8))
    display(control_portfolio.loc[['ts_only', 'metadata', 'text_residual', 'shuffled_content', 'text_only', 'buy_hold'], ['return_pct', 'bh_return_pct', 'excess_pct', 'alpha_annual_pct', 'beta']].round(5))
    display({'modo': control_mode, 'manchetes substituídas': int(mapping['changed'].sum()), 'manchetes não substituíveis': int((~mapping['changed'].astype(bool)).sum()), 'Δ MSE texto − embaralhado': float(control_signal.loc['text_residual', 'mse_5d'] - control_signal.loc['shuffled_content', 'mse_5d'])})
else:
    display({'modo': control_mode, 'resultados dos controles': 'pendentes; não inferidos nesta sessão'})

## 13. Regimes HMM: diagnóstico causal, sem otimizar pesos

O HMM de dois estados usa **apenas retornos de fechamento conhecidos** da cesta; ajusta parâmetros nas datas de treino e atualiza $P(z_t \mid r_{\leq t})$ em ordem cronológica. Não usa retorno futuro, alvo do modelo nem suavização Viterbi com todo o teste. O estado do dia $t$ só é conhecido **após o fechamento de $t$**; não se pode tratá-lo como sinal antes desse fechamento. Esta análise separa erros por regime, mas **não** escolhe regime, modelo, carteira ou hiperparâmetros usando o teste já visto. Estados raros e rótulos futuros sobrepostos limitam qualquer comparação.

In [ ]:
import importlib.util
if importlib.util.find_spec('hmmlearn') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'hmmlearn>=0.3.3'], check=True)
from src.fusion.presentation_regimes import analyze_regimes
regime_summary, regime_states = analyze_regimes(panel, {**forecasts, **control_forecasts})
regime_summary.to_csv(OUT / 'presentation_regime_summary.csv')
regime_states.to_csv(OUT / 'presentation_regime_states.csv', index=False)
regime_columns = ['regime', 'n_dates', 'n', 'n_with_news', 'headline_coverage', 'hmm_converged', 'mse_ts_only', 'mse_metadata', 'mse_text_residual', 'conditional_mse_gain_text_residual', 'ic_ts_only', 'ic_text_residual', 'caution']
if control_forecasts:
    regime_columns += ['mse_shuffled_content', 'mse_text_only', 'conditional_mse_gain_shuffled_content']
display(regime_summary[regime_columns].round(7))
display({'fit até': str(regime_states.attrs['train_end'].date()), 'dias de treino do HMM': regime_states.attrs['fit_days'], 'último estado no teste': regime_states.iloc[-1][['date', 'regime', 'state_probability']].to_dict()})

## 14. Período novo: preços reais disponíveis, notícias ainda não verificadas

Há **311 pregões com preços dos oito ativos entre 2024-01-02 e 2025-03-28** nos CSV locais; o pacote de preços e hashes pode ser lido do Drive. Isso **não** constitui teste novo: os arquivos de notícias existentes não cobrem janeiro de 2024, o GDELT devolveu HTTP 429 e a API histórica paga não está disponível. Ausência de feed significa cobertura **desconhecida**, não dia sem notícia. Os embeddings numéricos e textuais e os checkpoints de 2020–23 **não** cobrem este período e não podem produzir previsões aqui.

Para habilitar a coleta auditável, obtenha de um provedor com direito de uso oito arquivos `ADBE.jsonl` … `QCOM.jsonl` com `ticker`, `published_at` UTC com fuso, `source`, `url` HTTPS e `title`, mais `coverage.json` com provedor e janelas completas atestadas de 2024-01-02 a 2025-03-28. Coloque-os em `MyDrive/latent_fusion_data/verified_news_2024_2025/`. O script `src/fusion/new_period_news.py` valida relevância, datas, URLs e atraso de **um pregão adicional**, mantém todos os pregões inclusive sem manchete e bloqueia feeds incompletos; atestação do operador **não prova** completude. Só depois gere embeddings MOMENT/MiniLM em GPU no Colab e avalie uma arquitetura compatível treinada **sem reajustar no teste novo**. Este notebook não apresenta retornos fictícios para o período.

In [ ]:
PERIOD = DATA / 'presentation_new_period'
price_path = PERIOD / 'new_period_prices.npz'
report_path = PERIOD / 'coverage.json'
if not price_path.is_file() or not report_path.is_file():
    display({'período novo': 'artefato de preços ausente no Drive; exporte com build_new_period_package() no computador'})
else:
    with np.load(price_path, allow_pickle=False) as prices_new:
        price_tickers = prices_new['tickers'].tolist()
        price_dates = pd.DatetimeIndex(prices_new['dates'])
        price_close = prices_new['close']
        known_new = prices_new['known_return']
    price_report = json.loads(report_path.read_text(encoding='utf-8'))
    if (len(price_tickers) != 8 or price_close.shape != (8, len(price_dates)) or known_new.shape != price_close.shape
        or price_report['price_sessions'] != len(price_dates) or price_dates.has_duplicates or not price_dates.is_monotonic_increasing
        or price_report['start'] != str(price_dates.min().date()) or price_report['end'] != str(price_dates.max().date())
        or not np.isfinite(price_close).all() or not np.isnan(known_new[:, 0]).all()
        or not np.allclose(known_new[:, 1:], price_close[:, 1:] / price_close[:, :-1] - 1)):
        raise ValueError('Pacote de preços do período novo incompatível')
    NEWS_NEW = DATA / 'verified_news_2024_2025'
    if (NEWS_NEW / 'coverage.json').is_file():
        from src.fusion.new_period_news import build_new_period_package
        price_report = build_new_period_package(PERIOD, NEWS_NEW, price_dir=PERIOD / 'price_source')
    display(pd.DataFrame({'ativo': price_tickers, 'pregões': len(price_dates), 'primeira sessão': price_dates.min().date(), 'última sessão': price_dates.max().date(), 'status do feed': [price_report['tickers'][ticker]['feed_status'] for ticker in price_tickers]}).set_index('ativo'))
    display({'feed importado com atestação do operador': price_report['ready'], 'início': price_report['start'], 'fim': price_report['end'], 'fonte dos preços': price_report['price_source'], 'modelos e embeddings novos': 'não gerados; nenhum retorno pós-2023 estimado'})


## 15. O que pode ser apresentado — e o que não pode

**Implementação demonstrada:** atenção cruzada genuína, modalidade numérica congelada, correção textual condicional, ablação de metadados, três perfis simulados, custos, controle embaralhado e exclusivamente textual **somente quando seus resultados foram executados**, mais HMM ajustado apenas no treino com filtragem causal. **Resultado não demonstrado:** alfa atribuível ao significado das notícias ou desempenho em 2024–25. Os perfis e custos são cenários ilustrativos, não aconselhamento personalizado: todos mantêm 100% em oito ações, sem caixa, proteção de queda, impostos ou restrições de liquidez. O universo foi escolhido após observar a cobertura e o teste de 2022–23 já tinha sido inspecionado. O alvo de cinco pregões gera observações sobrepostas; manchete ausente não implica ausência de notícias; a fonte conhecida falha após novembro de 2023. O HMM **não altera pesos** nem prova que regimes melhoram o retorno.

**Resposta à banca:** se o texto não superar consistentemente idade/contagem, nem o controle com manchetes embaralhadas, não há evidência de que significado textual ajude neste conjunto. A arquitetura funcionar não prova ganho econômico. Próximo experimento: obter cobertura auditada e com direito de uso para **todo** o período novo, gerar embeddings em Colab e repetir as comparações sem reajustar hiperparâmetros no teste já visto.

In [ ]:
{'execução': mode,
 'controles': control_mode,
 'período do teste exploratório': (str(equity.index.min().date()), str(equity.index.max().date())),
 'texto acima de metadados no retorno líquido': bool(portfolio.loc['text_residual', 'return_pct'] > portfolio.loc['metadata', 'return_pct']),
 'texto acima de metadados no MSE': bool(signal.loc['text_residual', 'mse_5d'] < signal.loc['metadata', 'mse_5d']),
 'perfil exibido: retorno com texto (%)': round(float(comparison.loc[(PROFILE, 'text_residual'), 'return_pct']), 4),
 'perfil exibido: comprar e manter (%)': round(float(comparison.loc[(PROFILE, 'text_residual'), 'bh_return_pct']), 4),
 'novo período com notícias atestadas': price_report['ready'] if report_path.is_file() else False,
 'relatório original': str(OUT / 'results.md')}